# Benchmark Updates (Experiment 1) - gemini-3.8-flash

This notebook updates the original Experiment 1 benchmark (zero-shot) and benchmark + few-shot evaluations to use `gemini-3.8-flash`, for all three datasets used in the dissertation: **IBM**, **ABSRCT**, and **PECT** (AAE).

For each dataset, only two scenarios are evaluated:

1. **Zero-shot**: the plain prompt with no additional context (equivalent to "No Context" / "Sem Contexto" in the original Experiment 1 notebooks).
2. **Few-shot**: the same zero-shot task, prefixed with the same fixed worked examples used in the original Experiment 1 few-shot evaluation for that dataset.

The other Experiment 1 scenarios (Knowledge Graph context, internal sentiment context, Chain-of-Thought, and their combinations) are intentionally **not** repeated here.

The notebook is organized in three parts, one per dataset. Each part is split into a zero-shot and a few-shot subsection that builds the prompts, runs the batch evaluation, and reports accuracy / weighted F1-score.

In [ ]:
!pip install -q -U google-generativeai pandas scikit-learn tqdm

In [ ]:
import os
import pandas as pd
import google.generativeai as genai
from sklearn.metrics import accuracy_score, f1_score
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from google.colab import drive

In [ ]:
# ===============================================================================
# SHARED SETUP: API key, Google Drive, model, and helper functions
# ===============================================================================

GEMINI_API_KEY = ""  # Your Gemini API key here

try:
    genai.configure(api_key=GEMINI_API_KEY)
    print("Gemini API configured.")
except Exception as e:
    print(f"API key not found or invalid. Error: {e}")

try:
    drive.mount('/content/drive')
    print("Google Drive mounted.")
except Exception:
    print("Google Drive already mounted or error mounting.")

if not GEMINI_API_KEY:
    print("WARNING: Gemini API key not configured.")

# All scenarios in this notebook use the same evaluation model.
model = genai.GenerativeModel('models/gemini-3.8-flash')


def _call_gemini_api_with_retry(prompt, model):
    """Helper function that calls the API with retry logic.
    The prompt must already contain the full instruction."""
    for attempt in range(2):
        try:
            response = model.generate_content(prompt)
            answer = response.text.strip().upper()
            if "ATTACK" in answer:
                return "ATTACK"
            elif "SUPPORT" in answer:
                return "SUPPORT"
            else:
                return "N/A"
        except Exception as e:
            if attempt == 1:
                print(f"Final error processing prompt: {str(e)}")
                return "ERROR"
    return "ERROR"


def run_batch_evaluation(df, prompt_col, desc):
    """Runs _call_gemini_api_with_retry in parallel over every row of df,
    returning a dict {original_index: answer} that preserves df's pandas index
    (so it works both for a full DataFrame and for a single subject-group slice)."""
    results = {}
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {
            executor.submit(_call_gemini_api_with_retry, row[prompt_col], model): index
            for index, row in df.iterrows()
        }
        for future in tqdm(as_completed(futures), total=len(df), desc=desc):
            original_index = futures[future]
            try:
                results[original_index] = future.result()
            except Exception as e:
                print(f"Error in thread for index {original_index}: {e}")
                results[original_index] = "ERROR"
    return results


def build_zero_shot_prompt(argument, topic):
    """The plain benchmark prompt, with no additional context."""
    return (
        f"Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.\n\n"
        f"Question: The argument \"{argument}\" SUPPORT or ATTACK the belief \"{topic}\"?"
    )


def build_few_shot_prompt(argument, topic, few_shot_header):
    """The zero-shot task, prefixed with a fixed set of worked examples."""
    return (
        f"{few_shot_header}"
        f"Argument: \"{argument}\"\n"
        f"Belief: \"{topic}\"\n"
        f"Answer:"
    )


def print_metrics(df, pred_col, scenario_name):
    """Prints accuracy / weighted F1 for a single scenario column."""
    y_true = df['ground_truth_stance']
    y_pred = df[pred_col]
    valid_mask = df[pred_col].isin(['SUPPORT', 'ATTACK'])

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true[valid_mask], y_pred[valid_mask], average='weighted', zero_division=0)

    print(f"\n[Results for: {scenario_name}]")
    print(f"Accuracy: {acc:.4f} (including errors)")
    print(f"F1-Score (Weighted): {f1:.4f} (valid answers only)")


def print_metrics_by_group(df, pred_col, scenario_name, group_col='subject'):
    """Prints accuracy / weighted F1 for a single scenario column, broken down by group_col."""
    for group in df[group_col].unique():
        group_df = df[df[group_col] == group]
        y_true = group_df['ground_truth_stance']
        y_pred = group_df[pred_col]
        valid_mask = group_df[pred_col].isin(['SUPPORT', 'ATTACK'])

        if valid_mask.sum() == 0:
            print(f"\n[{scenario_name} - group '{group}']: no valid answers found.")
            continue

        acc = accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true[valid_mask], y_pred[valid_mask], average='weighted', zero_division=0)

        print(f"\n[Results for: {scenario_name} - group '{group}']")
        print(f"Accuracy: {acc:.4f} (including errors)")
        print(f"F1-Score (Weighted): {f1:.4f} (valid answers only)")

# Part 1 — IBM Dataset

## 1.1 Zero-shot prompt

In [ ]:
# ===============================================================================
# IBM - DATA LOADING AND PREPROCESSING (same split logic as Experiment 1)
# ===============================================================================
INPUT_CSV_PATH_IBM = ''          # Your claim_stance_dataset_v1.csv path here
OUTPUT_ZERO_SHOT_PATH_IBM = ''   # e.g. '.../results_IBM_zero_shot_38_benchmark_update.csv'
OUTPUT_FEW_SHOT_PATH_IBM = ''    # e.g. '.../results_IBM_few_shot_38_benchmark_update.csv'

df_ibm = pd.read_csv(INPUT_CSV_PATH_IBM)

df_ibm['claims.stance'] = df_ibm['claims.stance'].map({'PRO': 'SUPPORT', 'CON': 'ATTACK'})
df_ibm['argument'] = df_ibm['claims.claimCorrectedText']
df_ibm['topic'] = df_ibm['topicText']
df_ibm['ground_truth_stance'] = df_ibm['claims.stance']

test_df_ibm = df_ibm[df_ibm['split'] == 'test'].copy()
print(f"IBM test set loaded with {len(test_df_ibm)} rows.")

test_df_ibm['prompt_zero_shot'] = test_df_ibm.apply(
    lambda row: build_zero_shot_prompt(row['argument'], row['topic']), axis=1
)

print("\n--- Example zero-shot prompt (IBM) ---")
print(test_df_ibm['prompt_zero_shot'].iloc[0])

In [ ]:
# ===============================================================================
# IBM - ZERO-SHOT BATCH EVALUATION
# ===============================================================================
results_zero_shot_ibm = run_batch_evaluation(test_df_ibm, 'prompt_zero_shot', "Evaluating IBM zero-shot")
test_df_ibm['gemini_zero_shot'] = pd.Series(results_zero_shot_ibm)

test_df_ibm.to_csv(OUTPUT_ZERO_SHOT_PATH_IBM, index=False, encoding='utf-8')
print(f"\n--- Evaluation completed. Results saved to '{OUTPUT_ZERO_SHOT_PATH_IBM}' ---")

In [ ]:
print_metrics(test_df_ibm, 'gemini_zero_shot', "IBM - Zero-shot (gemini-3.8-flash)")

## 1.2 Few-shot prompt (same examples as the original Experiment 1 few-shot evaluation)

In [ ]:
# ===============================================================================
# IBM - FEW-SHOT PROMPT CONSTRUCTION (same examples used in Experiment 1)
# ===============================================================================
ibm_example_1 = (
    "Argument: \"Marketing promotes consumerism and waste.\"\n"
    "Belief: \"This house believes that advertising is harmful.\"\n"
    "Answer: SUPPORT"
)
ibm_example_2 = (
    "Argument: \"Unity is seen as an essential feature of the nation and the nation-state.\"\n"
    "Belief: \"This house would embrace multiculturalism.\"\n"
    "Answer: ATTACK"
)
ibm_example_3 = (
    "Argument: \"Children with many siblings receive fewer resources.\"\n"
    "Belief: \"This house supports the one-child policy of the republic of China.\"\n"
    "Answer: SUPPORT"
)
ibm_few_shot_header = (
    f"Here are some examples:\n\n[EXAMPLE 1]\n{ibm_example_1}\n\n[EXAMPLE 2]\n{ibm_example_2}\n\n"
    f"[EXAMPLE 3]\n{ibm_example_3}\n\n---\n\nNow, complete the following task:\n"
)

test_df_ibm['prompt_few_shot'] = test_df_ibm.apply(
    lambda row: build_few_shot_prompt(row['argument'], row['topic'], ibm_few_shot_header), axis=1
)

print("\n--- Example few-shot prompt (IBM) ---")
print(test_df_ibm['prompt_few_shot'].iloc[0])

In [ ]:
results_few_shot_ibm = run_batch_evaluation(test_df_ibm, 'prompt_few_shot', "Evaluating IBM few-shot")
test_df_ibm['gemini_few_shot'] = pd.Series(results_few_shot_ibm)

test_df_ibm.to_csv(OUTPUT_FEW_SHOT_PATH_IBM, index=False, encoding='utf-8')
print(f"\n--- Evaluation completed. Results saved to '{OUTPUT_FEW_SHOT_PATH_IBM}' ---")

In [ ]:
print_metrics(test_df_ibm, 'gemini_few_shot', "IBM - Few-shot (gemini-3.8-flash)")

# Part 2 — ABSRCT Dataset

## 2.1 Zero-shot prompt

In [ ]:
# ===============================================================================
# ABSRCT - DATA LOADING
# ===============================================================================
INPUT_CSV_PATH_ABSRCT = ''          # Your AbstRT.csv dataset path here
OUTPUT_ZERO_SHOT_PATH_ABSRCT = ''   # e.g. '.../results_ABSRCT_zero_shot_38_benchmark_update.csv'
OUTPUT_FEW_SHOT_PATH_ABSRCT = ''    # e.g. '.../results_ABSRCT_few_shot_38_benchmark_update.csv'

df_absrct = pd.read_csv(INPUT_CSV_PATH_ABSRCT)
print(f"ABSRCT dataset loaded with {len(df_absrct)} rows.")
print(df_absrct['subject'].value_counts())

df_absrct['prompt_zero_shot'] = df_absrct.apply(
    lambda row: build_zero_shot_prompt(row['argument'], row['topic']), axis=1
)

print("\n--- Example zero-shot prompt (ABSRCT) ---")
print(df_absrct['prompt_zero_shot'].iloc[0])

In [ ]:
# ===============================================================================
# ABSRCT - ZERO-SHOT BATCH EVALUATION (per subject group)
# ===============================================================================
results_zero_shot_absrct = {}
for group in df_absrct['subject'].unique():
    group_df = df_absrct[df_absrct['subject'] == group]
    results_zero_shot_absrct.update(
        run_batch_evaluation(group_df, 'prompt_zero_shot', f"Evaluating ABSRCT zero-shot '{group}'")
    )

df_absrct['gemini_zero_shot'] = pd.Series(results_zero_shot_absrct)
df_absrct.to_csv(OUTPUT_ZERO_SHOT_PATH_ABSRCT, index=False, encoding='utf-8')
print(f"\n--- Evaluation completed. Results saved to '{OUTPUT_ZERO_SHOT_PATH_ABSRCT}' ---")

In [ ]:
print_metrics_by_group(df_absrct, 'gemini_zero_shot', "ABSRCT - Zero-shot (gemini-3.8-flash)")

## 2.2 Few-shot prompt (same examples as the original Experiment 1 few-shot evaluation)

In [ ]:
# ===============================================================================
# ABSRCT - FEW-SHOT PROMPT CONSTRUCTION (same examples used in Experiment 1)
# ===============================================================================
absrct_example_1 = (
    "Argument: \"In the context of chronic physical illness, such as breast cancer, depression is associated with increased morbidity, longer periods of hospitalization, and greater overall disability.\"\n"
    "Belief: \"Outpatient treatment with suramin plus HC is well tolerated and provides moderate palliative benefit and delay in disease progression for patients with symptomatic HRPC.\"\n"
    "Answer: SUPPORT"
)
absrct_example_2 = (
    "Argument: \"Older patients, even if fit, are often considered incapable of tolerating platinum-based systemic therapy.\"\n"
    "Belief: \"Advanced age alone should not preclude appropriate NSCLC treatment.\"\n"
    "Answer: ATTACK"
)
absrct_few_shot_header = (
    f"Here are some examples:\n\n[EXAMPLE 1]\n{absrct_example_1}\n\n[EXAMPLE 2]\n{absrct_example_2}\n\n"
    f"---\n\nNow, complete the following task:\n"
)

df_absrct['prompt_few_shot'] = df_absrct.apply(
    lambda row: build_few_shot_prompt(row['argument'], row['topic'], absrct_few_shot_header), axis=1
)

print("\n--- Example few-shot prompt (ABSRCT) ---")
print(df_absrct['prompt_few_shot'].iloc[0])

In [ ]:
results_few_shot_absrct = {}
for group in df_absrct['subject'].unique():
    group_df = df_absrct[df_absrct['subject'] == group]
    results_few_shot_absrct.update(
        run_batch_evaluation(group_df, 'prompt_few_shot', f"Evaluating ABSRCT few-shot '{group}'")
    )

df_absrct['gemini_few_shot'] = pd.Series(results_few_shot_absrct)
df_absrct.to_csv(OUTPUT_FEW_SHOT_PATH_ABSRCT, index=False, encoding='utf-8')
print(f"\n--- Evaluation completed. Results saved to '{OUTPUT_FEW_SHOT_PATH_ABSRCT}' ---")

In [ ]:
print_metrics_by_group(df_absrct, 'gemini_few_shot', "ABSRCT - Few-shot (gemini-3.8-flash)")

# Part 3 — PECT Dataset

## 3.1 Zero-shot prompt

In [ ]:
# ===============================================================================
# PECT - DATA LOADING
# ===============================================================================
INPUT_CSV_PATH_PECT = ''          # Your aae_test.csv dataset path here
OUTPUT_ZERO_SHOT_PATH_PECT = ''   # e.g. '.../results_PECT_zero_shot_38_benchmark_update.csv'
OUTPUT_FEW_SHOT_PATH_PECT = ''    # e.g. '.../results_PECT_few_shot_38_benchmark_update.csv'

df_pect = pd.read_csv(INPUT_CSV_PATH_PECT)
if 'classification' in df_pect.columns and 'ground_truth_stance' not in df_pect.columns:
    df_pect.rename(columns={'classification': 'ground_truth_stance'}, inplace=True)
print(f"PECT dataset loaded with {len(df_pect)} rows.")

df_pect['prompt_zero_shot'] = df_pect.apply(
    lambda row: build_zero_shot_prompt(row['argument'], row['topic']), axis=1
)

print("\n--- Example zero-shot prompt (PECT) ---")
print(df_pect['prompt_zero_shot'].iloc[0])

In [ ]:
# ===============================================================================
# PECT - ZERO-SHOT BATCH EVALUATION
# ===============================================================================
results_zero_shot_pect = run_batch_evaluation(df_pect, 'prompt_zero_shot', "Evaluating PECT zero-shot")
df_pect['gemini_zero_shot'] = pd.Series(results_zero_shot_pect)

df_pect.to_csv(OUTPUT_ZERO_SHOT_PATH_PECT, index=False, encoding='utf-8')
print(f"\n--- Evaluation completed. Results saved to '{OUTPUT_ZERO_SHOT_PATH_PECT}' ---")

In [ ]:
print_metrics(df_pect, 'gemini_zero_shot', "PECT - Zero-shot (gemini-3.8-flash)")

## 3.2 Few-shot prompt (same examples as the original Experiment 1 few-shot evaluation)

In [ ]:
# ===============================================================================
# PECT - FEW-SHOT PROMPT CONSTRUCTION (same examples used in Experiment 1)
# ===============================================================================
pect_example_1 = (
    "Argument: \"What we acquired from team work is not only how to achieve the same goal with others but more importantly, how to get along with other\"\n"
    "Belief: \"through cooperation, children can learn about interpersonal skills which are significant in the future life of all students.\"\n"
    "Answer: SUPPORT"
)
pect_example_2 = (
    "Argument: \"competition makes the society more effective\"\n"
    "Belief: \"we should attach more importance to cooperation during primary education\"\n"
    "Answer: ATTACK"
)
pect_few_shot_header = (
    f"Here are some examples:\n\n[EXAMPLE 1]\n{pect_example_1}\n\n[EXAMPLE 2]\n{pect_example_2}\n\n"
    f"---\n\nNow, complete the following task:\n"
)

df_pect['prompt_few_shot'] = df_pect.apply(
    lambda row: build_few_shot_prompt(row['argument'], row['topic'], pect_few_shot_header), axis=1
)

print("\n--- Example few-shot prompt (PECT) ---")
print(df_pect['prompt_few_shot'].iloc[0])

In [ ]:
results_few_shot_pect = run_batch_evaluation(df_pect, 'prompt_few_shot', "Evaluating PECT few-shot")
df_pect['gemini_few_shot'] = pd.Series(results_few_shot_pect)

df_pect.to_csv(OUTPUT_FEW_SHOT_PATH_PECT, index=False, encoding='utf-8')
print(f"\n--- Evaluation completed. Results saved to '{OUTPUT_FEW_SHOT_PATH_PECT}' ---")

In [ ]:
print_metrics(df_pect, 'gemini_few_shot', "PECT - Few-shot (gemini-3.8-flash)")